In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import json

In [98]:
url = "https://www.youtube.com/watch?v=MVpOKwTKVSc"
html = requests.get(url).text

data = re.search(r"ytInitialData = ({.*?});", html).group(1)
json_data = json.loads(data)

markers = json_data["playerOverlays"]["playerOverlayRenderer"] \
    ["decoratedPlayerBarRenderer"]["decoratedPlayerBarRenderer"] \
    ["playerBar"]["multiMarkersPlayerBarRenderer"]["markersMap"]

# find AUTO_CHAPTERS
chapters_data = None
for item in markers:
    if item["key"] == "AUTO_CHAPTERS":
        chapters_data = item["value"]["chapters"]

# extract clean result
chapters = []
for ch in chapters_data:
    title = ch["chapterRenderer"]["title"]["simpleText"]
    time_ms = ch["chapterRenderer"]["timeRangeStartMillis"]
    time_sec = time_ms // 1000

    chapters.append({
        "title": title,
        "start_seconds": time_sec
    })

print(chapters)
print("Number of chapters:", len(chapters))

[{'title': 'Introduction', 'start_seconds': 0}, {'title': 'Binary Division', 'start_seconds': 28}, {'title': 'Division Algorithms', 'start_seconds': 401}]
Number of chapters: 3


In [ ]:
url = "https://www.youtube.com/watch?v=MVpOKwTKVSc"
headers = {"User-Agent": "Mozilla/5.0"}

html = requests.get(url, headers=headers).text
soup = BeautifulSoup(html, "html.parser")


def extract_json(key):
    for script in soup.find_all("script"):
        if key in script.text:
            match = re.search(rf'{key}\s*=\s*({{.*}});', script.text)
            if match:
                try:
                    return json.loads(match.group(1))
                except:
                    pass
    return None


initial_data = extract_json("ytInitialData")
player_data = extract_json("ytInitialPlayerResponse")

chapters = []

try:
    markers = player_data["playerOverlays"]["playerOverlayRenderer"] \
        ["decoratedPlayerBarRenderer"]["decoratedPlayerBarRenderer"] \
        ["playerBar"]["multiMarkersPlayerBarRenderer"]["markersMap"]

    for item in markers:
        if item["key"] == "AUTO_CHAPTERS":
            for ch in item["value"]["chapters"]:
                chapters.append({
                    "title": ch["chapterRenderer"]["title"]["simpleText"],
                    "start_seconds": ch["chapterRenderer"]["timeRangeStartMillis"] // 1000
                })

except (KeyError, TypeError):
    pass

if not chapters:
    try:
        markers = initial_data["playerOverlays"]["playerOverlayRenderer"] \
            ["decoratedPlayerBarRenderer"]["decoratedPlayerBarRenderer"] \
            ["playerBar"]["multiMarkersPlayerBarRenderer"]["markersMap"]

        # find AUTO_CHAPTERS
        chapters_data = None
        for item in markers:
            if item["key"] == "AUTO_CHAPTERS":
                chapters_data = item["value"]["chapters"]

        for ch in chapters_data:
            title = ch["chapterRenderer"]["title"]["simpleText"]
            time_ms = ch["chapterRenderer"]["timeRangeStartMillis"]
            time_sec = time_ms // 1000

            chapters.append({
                "title": title,
                "start_seconds": time_sec
            })

    except (KeyError, TypeError):
        pass
        

if not chapters:
    try:
        description = player_data["videoDetails"]["shortDescription"]

        matches = re.findall(r'(\d{1,2}:\d{2})\s+(.+)', description)

        for time_str, title in matches:
            minutes, seconds = map(int, time_str.split(":"))
            total_seconds = minutes * 60 + seconds

            chapters.append({
                "title": title.strip(),
                "start_seconds": total_seconds
            })

    except (KeyError, TypeError):
        pass


if chapters:
    print("Chapters found:")
    print(chapters)
else:
    print("No chapters available for this video")

Chapters found:
[{'title': 'Introduction', 'start_seconds': 0}, {'title': 'Binary Division', 'start_seconds': 28}, {'title': 'Division Algorithms', 'start_seconds': 401}]


In [101]:
len(chapters)

3